# 09 — Error Analysis and Research Interpretation

This notebook supports the proposal's manual inspection requirement. It analyzes correct answers, correct abstentions, false answers and unnecessary refusals, and helps turn the experimental outputs into defensible report conclusions.

## 1. Load test predictions and evaluation results

Manual inspection is part of the proposal. We categorize outcomes into correct answers, correct abstentions, false answers and unnecessary refusals. We also inspect retrieval failures and QA failures.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT=Path.cwd()
if ROOT.name=="notebooks": ROOT=ROOT.parent
SPLITS=ROOT/"data"/"splits"
RESULTS=ROOT/"results"

test=pd.read_csv(SPLITS/"test.csv")
test["answers"]=test["answers"].apply(json.loads)
test["is_answerable"]=test["is_answerable"].astype(bool)
for filename in ["test_retrieval_features.csv","test_qa_predictions.csv",
                 "test_threshold_decisions.csv","test_classifier_predictions.csv"]:
    print(filename, (RESULTS/filename).exists())

ret=pd.read_csv(RESULTS/"test_retrieval_features.csv")
qa=pd.read_csv(RESULTS/"test_qa_predictions.csv")
thr=pd.read_csv(RESULTS/"test_threshold_decisions.csv")
clf=pd.read_csv(RESULTS/"test_classifier_predictions.csv")
df=test.merge(ret,on="question_id").merge(qa,on="question_id").merge(
    thr,on="question_id").merge(clf,on="question_id")
display(df.head())

## 2. Define outcome categories

For the classifier strategy:
- **Correct answer decision:** answerable and classifier says answer.
- **Correct abstention:** unanswerable and classifier says abstain.
- **False answer:** unanswerable and classifier says answer.
- **Unnecessary refusal:** answerable and classifier says abstain.

These categories evaluate the answerability decision. We separately inspect whether the generated answer matches a reference answer.

In [ ]:
def outcome(row):
    if row.is_answerable and row.classifier_answer:
        return "Correct answer decision"
    if (not row.is_answerable) and (not row.classifier_answer):
        return "Correct abstention"
    if (not row.is_answerable) and row.classifier_answer:
        return "False answer"
    return "Unnecessary refusal"

df["classifier_outcome"]=df.apply(outcome,axis=1)
display(df.classifier_outcome.value_counts().to_frame("count"))

## 3. Inspect examples by outcome

Read the question, reference answer, predicted answer and retrieved score. For each example, ask:
- Did BM25 retrieve the source passage?
- Does the retrieved evidence contain the answer?
- Did the QA model extract the correct span?
- Was the abstention decision reasonable?
- Is the original SQuAD label an adequate proxy for the retrieved evidence?

The final question is especially important for interpreting retrieval-based errors.

In [ ]:
def inspect_category(category, n=5, seed=42):
    sample=df[df.classifier_outcome==category].sample(
        n=min(n,(df.classifier_outcome==category).sum()),random_state=seed
    )
    for r in sample.itertuples():
        print("="*100)
        print("Outcome:",r.classifier_outcome)
        print("Question ID:",r.question_id)
        print("Article:",r.article_title)
        print("Question:",r.question)
        print("Gold answerable in paired passage:",r.is_answerable)
        print("Reference answer:",r.answer_text)
        print("Predicted answer:",r.predicted_answer)
        print("BM25 top score:",r.top_bm25_score)
        print("Source passage retrieved:",r.source_passage_retrieved)
        print("QA score:",r.qa_score)
        print()

for category in [
    "Correct answer decision","Correct abstention",
    "False answer","Unnecessary refusal"
]:
    print("\n###",category)
    inspect_category(category,n=3)

## 4. Retrieval failure analysis

Measure how often the paired source passage appears in the top-k results, split by the original answerability label. A missing source passage can explain why a QA model fails even when the original passage contains an answer.

In [ ]:
retrieval_summary=df.groupby("is_answerable").source_passage_retrieved.agg(
    ["count","mean"]
).rename(index={True:"Answerable",False:"Unanswerable"})
display(retrieval_summary)

## 5. Save a manually reviewable error sample

Export a compact CSV containing the classifier's false answers and unnecessary refusals. Add your own columns in a spreadsheet or manually reviewed notebook table: `reviewed`, `error_cause`, `notes`. Do not treat the automatic category as a complete explanation.

In [ ]:
error_sample=df[df.classifier_outcome.isin(
    ["False answer","Unnecessary refusal"]
)][[
    "question_id","article_title","question","context",
    "is_answerable","answer_text","predicted_answer",
    "top_bm25_score","source_passage_retrieved",
    "qa_score","answerability_probability","classifier_outcome"
]].copy()
error_sample["reviewed"]=""
error_sample["error_cause"]=""
error_sample["notes"]=""
error_sample.to_csv(RESULTS/"manual_error_review.csv",index=False)
print("Saved",len(error_sample),"errors for manual review.")

## 6. Prepare conclusions for the report

Use the evidence from the notebooks to answer:

1. Did abstention reduce unsupported answers compared with always-answer?
2. What happened to the false-abstention rate?
3. Did the classifier outperform a single BM25 threshold?
4. How did retrieval failures affect answer quality?
5. Which examples reveal limitations of the original SQuAD labels as retrieval-evidence labels?
6. What would you change with more time (semantic retrieval, better evidence labels, reranking, calibration, larger manually reviewed test set)?

### Reproducibility checklist

- [ ] Dataset source and version recorded.
- [ ] Random seed recorded.
- [ ] Article-level split verified.
- [ ] Balanced subset counts documented.
- [ ] BM25 top-k held fixed.
- [ ] Thresholds selected on validation only.
- [ ] Test set evaluated once after decisions were fixed.
- [ ] Model checkpoint and library versions recorded.
- [ ] Error examples manually inspected.
- [ ] Limitations and label assumptions explained.

**End of experimental notebook sequence.**